In [1]:
import pandas as pd
import sklearn as sk

In [2]:
df = pd.read_csv('../data/sampled_metadata_aux.csv')
df.head(3)

,patient_id,study_id,dataset_id,patient_sex,age,machine,number_of_slices,slice_thickness,pixel_spacing_xy,pixel_spacing_z,...,Achados,window_center,window_width,volumetric,x,y,z,Category,local_Path,selected_local
0,2332,"1,28E+73",jpr_dados,F,28.0,TOSHIBA,207.0,1.00,0.449000,0.8,...,Normal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,D:/Backup/Mestrado/dados/JPR/dataset_jpr_train...,True
1,1636,"1,28E+73",jpr_dados,F,52.0,Philips,229.0,1.50,0.431641,0.75,...,Normal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,D:/Backup/Mestrado/dados/JPR/dataset_jpr_train...,True
2,2605,"1,28E+73",jpr_dados,F,58.0,GE MEDICAL SYSTEMS,279.0,1.25,0.419922,0.625,...,Normal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,D:/Backup/Mestrado/dados/JPR/dataset_jpr_train...,True


In [3]:
import os


def transform_path(path):
  if not isinstance(path, str):
    return path

  # Normalize slashes
  norm_path = path.replace("\\", "/")

  # Case 1: Physionet (extracts ID before .nii and adds trailing slash)
  if "dados/Physionet/physionet_scans/ct_scans/" in norm_path:
    patient_id = os.path.splitext(os.path.basename(norm_path))[0]
    return f"/home/jovyan/imgs/png_data/physionet/{patient_id}/"

  # Case 2: JPR (extracts the last folder name/ID)
  if "dados/JPR/dataset_jpr_train/dataset_jpr_train/" in norm_path:
    patient_id = os.path.basename(norm_path.rstrip("/"))
    return f"/home/jovyan/imgs/png_data/jpr_dados/{patient_id}"

  return norm_path


# Apply to the dataframe
df["png_Path"] = df["local_Path"].apply(transform_path)

Das 776 imagens normais, selecionei 200 para ficarem no conjunto de teste. df_test e df_rest são definidas assim para que não haja data leakage.

In [4]:
df_normal = df[df['Achados'] == 'Normal']
# 200 imagens saudáveis aleatórias + todas imagens com algum achado
df_test = pd.concat([df_normal.sample(200, random_state = 42), df[df['Achados'] != 'Normal']])
# df_test + df_rest não vão dar o mesmo número de imagens para que não haja patient leakage
df_rest = df[~df['patient_id'].isin(df_test['patient_id'])]

In [5]:
train_df, val_df = sk.model_selection.train_test_split(df_rest)

In [6]:
train_df.to_csv('../data/splits/train_data.csv', index=False)
val_df.to_csv('../data/splits/val_data.csv', index=False)
df_test.to_csv('../data/splits/test_data.csv', index=False)